# GPU EDA Persisted Outputs -> Parquet Export Notebook (v1.0.0)

This notebook is the contract bridge between persisted Spark/Nessie EDA outputs and the local visualization notebook. It serves as a lightweight pass-through layer that exports persisted tables as Parquet with proper metadata for downstream consumption.

## Notebook Pipeline Position

1. `dcgm_raw_impute_sparkcaster_1.1.ipynb` - Builds engineered features (including region_rollup), imputes missing values, writes cleaned tables
2. `dcgm_eda_sparkcaster_1.2.4.ipynb` - Reads cleaned tables, computes segmented EDA outputs, persists analysis tables
3. **This notebook** - Exports persisted EDA tables to Parquet with manifest and validation
4. `dcgm_eda_local_visualizations_monthly_1.0.0.ipynb` - Reads Parquet export, renders segmented charts and HTML report

## What This Notebook Exports

- `raw_sample` - With segment source columns (gen, region_rollup, customer_segment, product_segment, is_training)
- `corr_summary` - With segment_dim/segment_value for segmented correlations
- `corr_over_time` - Time-based correlations with segmentation support
- `density_bins` - 2D density bins with segmentation
- `density_over_time` - Time-based density with segmentation
- `bin_metadata` - Metadata for binning operations
- `trend_summary` - Metric trends over time with segmentation

## Key Features

- **Schema Validation**: Validates presence of required segmentation columns
- **Enhanced Manifest**: Includes column lists, segmentation metadata, and validation results
- **Time Enrichment**: Adds `month_period` column to time-based datasets for monthly visualizations
- **Backward Compatible**: Handles both segmented and non-segmented data gracefully
- **Export Contract**: Clear contract for downstream consumption via manifest.json

## Segmentation Support

The notebook preserves segmentation columns from upstream:
- **Segment dimensions**: gen, region_rollup, customer_segment, product_segment, is_training
- **Generalized shape**: segment_dim and segment_value columns for flexible segmentation

## Configuration

- `EXPORT_ROOT`: Base directory for Parquet exports (default: /home/coreweave/dcgm_eda_parquet)
- `COMPRESSION`: Parquet compression (default: snappy)
- `OVERWRITE`: Whether to overwrite existing exports (default: True)

In [11]:

# ========================================
# PACKAGE INSTALLATION
# ========================================
import importlib
import importlib.metadata as importlib_metadata
import subprocess
import sys
import site


def is_module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False


def install_if_missing(pip_name: str, import_name: str = None):
    import_name = import_name or pip_name
    dist_name = pip_name.split('==', 1)[0]
    required_version = pip_name.split('==', 1)[1] if '==' in pip_name else None

    if not is_module_available(import_name):
        print(f'Installing {pip_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', pip_name])
        return

    if required_version:
        try:
            installed_version = importlib_metadata.version(dist_name)
        except importlib_metadata.PackageNotFoundError:
            installed_version = None
        if installed_version != required_version:
            print(f'Upgrading {dist_name} from {installed_version or "unknown"} to {required_version} ...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '--upgrade', pip_name])
            return

    print(f'✓ {pip_name} already installed')


print('=' * 60)
print('INSTALLING EXPORT NOTEBOOK DEPENDENCIES')
print('=' * 60)

install_if_missing('keyring', 'keyring')
install_if_missing('keyrings.cryptfile', 'keyrings.cryptfile')
install_if_missing('pyarrow', 'pyarrow')
install_if_missing('pandas', 'pandas')
install_if_missing('fsspec', 'fsspec')
install_if_missing('s3fs', 's3fs')

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

print('✓ Dependency validation complete')
print('=' * 60)


INSTALLING EXPORT NOTEBOOK DEPENDENCIES
✓ keyring already installed
✓ keyrings.cryptfile already installed
✓ pyarrow already installed
✓ pandas already installed
✓ fsspec already installed
✓ s3fs already installed
✓ Dependency validation complete


In [12]:

# ========================================
# CAIOS CREDENTIALS SETUP
# ========================================
import keyring
import os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

print('=' * 60)
print('CAIOS CREDENTIALS SETUP')
print('=' * 60)

home_dir = str(Path.home())
os.environ['KEYRING_CRYPTFILE_PATH'] = f'{home_dir}/.local/share/python_keyring/cryptfile_pass.cfg'

kr = CryptFileKeyring()
kr.keyring_key = getpass('Set/enter master password for encrypted keyring: ')
keyring.set_keyring(kr)

caios_access_key = keyring.get_password('caios', 'access_key')
caios_secret_key = keyring.get_password('caios', 'secret_key')

if not caios_access_key:
    caios_access_key = input('Enter CAIOS access key: ')
    keyring.set_password('caios', 'access_key', caios_access_key)

if not caios_secret_key:
    caios_secret_key = getpass('Enter CAIOS secret key: ')
    keyring.set_password('caios', 'secret_key', caios_secret_key)

print('✓ CAIOS credentials configured')
print('=' * 60)


CAIOS CREDENTIALS SETUP
✓ CAIOS credentials configured


In [13]:

# ========================================
# SPARKCASTER + NESSIE INITIALIZATION
# ========================================
from spark.nessie.client import NessieSparkClient
from pyspark.sql import SparkSession

print('=' * 60)
print('SPARKCASTER + NESSIE INITIALIZATION')
print('=' * 60)

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        print('Stopping existing Spark session...')
        existing_spark.stop()
        print('✓ Existing session stopped')
except Exception as e:
    print(f'Note: {e}')

ness = NessieSparkClient(
    svc_url='http://kf-proxy.nessie.svc.cluster.local:19120/api/v2',
    nessie_endpoint='http://nessie-prd.cwobject.com',
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)

spark = ness.spark
spark.sparkContext.setLogLevel('ERROR')

print('✓ Sparkcaster + Nessie ready')
print(f'Spark Version: {spark.version}')
print(f'App Name: {spark.sparkContext.appName}')
print('=' * 60)


SPARKCASTER + NESSIE INITIALIZATION
Stopping existing Spark session...
✓ Existing session stopped
✓ Sparkcaster + Nessie ready
Spark Version: 3.5.5
App Name: pyspark-shell


In [14]:

# ========================================
# RUNTIME CONFIGURATION
# ========================================
from datetime import datetime
from pathlib import Path
from pyspark.sql import functions as F

print('=' * 60)
print('RUNTIME CONFIGURATION')
print('=' * 60)

OUTPUT_SCHEMA = 'sandbox.sandbox_finance'
OUTPUT_PREFIX = 'dcgm_eda'

TABLES = {
    'raw_sample': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_raw_sample',
    'corr_summary': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_summary',
    'corr_over_time': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_corr_over_time',
    'density_bins': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_bins',
    'density_over_time': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_density_over_time',
    'bin_metadata': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_bin_metadata',
    'trend_summary': f'{OUTPUT_SCHEMA}.{OUTPUT_PREFIX}_trend_summary',
}

TIME_COLUMN_MAP = {
    'raw_sample': 'datestamp',
    'corr_over_time': 'time_bucket',
    'density_over_time': 'time_bucket',
    'trend_summary': 'time_bucket',
}

EXPORT_ROOT = '/home/coreweave/dcgm_eda_parquet'
EXPORT_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OVERWRITE = True
COMPRESSION = 'snappy'

export_root = Path(EXPORT_ROOT) / EXPORT_TAG
manifest_path = export_root / 'manifest.json'

print('Persisted tables to export:')
for name, table_name in TABLES.items():
    print(f'  {name:18} -> {table_name}')

print('Export settings:')
print(f'  EXPORT_ROOT: {EXPORT_ROOT}')
print(f'  EXPORT_TAG : {EXPORT_TAG}')
print(f'  COMPRESSION: {COMPRESSION}')
print(f'  OVERWRITE  : {OVERWRITE}')
print('=' * 60)


RUNTIME CONFIGURATION
Persisted tables to export:
  raw_sample         -> sandbox.sandbox_finance.dcgm_eda_raw_sample
  corr_summary       -> sandbox.sandbox_finance.dcgm_eda_corr_summary
  corr_over_time     -> sandbox.sandbox_finance.dcgm_eda_corr_over_time
  density_bins       -> sandbox.sandbox_finance.dcgm_eda_density_bins
  density_over_time  -> sandbox.sandbox_finance.dcgm_eda_density_over_time
  bin_metadata       -> sandbox.sandbox_finance.dcgm_eda_bin_metadata
  trend_summary      -> sandbox.sandbox_finance.dcgm_eda_trend_summary
Export settings:
  EXPORT_ROOT: /home/coreweave/dcgm_eda_parquet
  EXPORT_TAG : 20260625_164113
  COMPRESSION: snappy
  OVERWRITE  : True


In [15]:

# ========================================
# LOAD PERSISTED TABLES
# ========================================
print('=' * 60)
print('LOADING PERSISTED TABLES')
print('=' * 60)

datasets = {}
for name, table_name in TABLES.items():
    print(f'Loading {name}: {table_name}')
    datasets[name] = spark.table(table_name)
    print('  ✓ readable')

print('✓ All persisted tables loaded')
print('=' * 60)


LOADING PERSISTED TABLES
Loading raw_sample: sandbox.sandbox_finance.dcgm_eda_raw_sample
  ✓ readable
Loading corr_summary: sandbox.sandbox_finance.dcgm_eda_corr_summary
  ✓ readable
Loading corr_over_time: sandbox.sandbox_finance.dcgm_eda_corr_over_time
  ✓ readable
Loading density_bins: sandbox.sandbox_finance.dcgm_eda_density_bins
  ✓ readable
Loading density_over_time: sandbox.sandbox_finance.dcgm_eda_density_over_time
  ✓ readable
Loading bin_metadata: sandbox.sandbox_finance.dcgm_eda_bin_metadata
  ✓ readable
Loading trend_summary: sandbox.sandbox_finance.dcgm_eda_trend_summary
  ✓ readable
✓ All persisted tables loaded


In [16]:
# ========================================
# EXPORT HELPERS
# ========================================
import json
from pathlib import Path


def add_month_period(df, time_col: str):
    return df.withColumn('month_period', F.date_format(F.to_timestamp(F.col(time_col)), 'yyyy-MM'))


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    return path


def export_dataset(df, output_path: Path, overwrite: bool = True):
    mode = 'overwrite' if overwrite else 'errorifexists'
    df.write.mode(mode).option('compression', COMPRESSION).parquet(str(output_path))


def preview_schema(df):
    fields = [f'{field.name}:{field.dataType.simpleString()}' for field in df.schema.fields[:12]]
    preview = ', '.join(fields)
    if len(df.schema.fields) > 12:
        preview += f', ... ({len(df.schema.fields)} total columns)'
    return preview


def get_segment_columns(df):
    """Identify segmentation-related columns in the dataframe."""
    segment_source_cols = ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
    segment_dim_cols = ['segment_dim', 'segment_value']

    found_source = [col for col in segment_source_cols if col in df.columns]
    found_dim = [col for col in segment_dim_cols if col in df.columns]

    return {
        'source_columns': found_source,
        'dimension_columns': found_dim,
        'has_segmentation': len(found_source) > 0 or len(found_dim) > 0
    }


def validate_schema_completeness(df, dataset_name: str):
    """Validate that required columns exist for each dataset type."""
    validation_results = {
        'dataset': dataset_name,
        'valid': True,
        'issues': []
    }

    segment_info = get_segment_columns(df)

    # Check raw_sample for required segment source fields
    if dataset_name == 'raw_sample':
        required_cols = ['gen', 'region_rollup', 'customer_segment', 'product_segment', 'is_training']
        missing = [col for col in required_cols if col in df.columns]
        if len(missing) < len(required_cols):
            not_found = [col for col in required_cols if col not in df.columns]
            # Not an error if columns don't exist yet (backward compatibility)
            validation_results['issues'].append(f'Info: segment source columns not yet present: {not_found}')

    # Check segmented datasets for segment_dim and segment_value
    if dataset_name in ['corr_summary', 'corr_over_time', 'density_bins', 'density_over_time', 'trend_summary']:
        if 'segment_dim' in df.columns or 'segment_value' in df.columns:
            if not ('segment_dim' in df.columns and 'segment_value' in df.columns):
                validation_results['issues'].append('Warning: partial segmentation columns (need both segment_dim and segment_value)')
                validation_results['valid'] = False

    return validation_results, segment_info


ensure_dir(export_root)
print(f'Export root ready: {export_root}')

Export root ready: /home/coreweave/dcgm_eda_parquet/20260625_164113


In [17]:
# ========================================
# EXPORT PARQUET DATASETS WITH VALIDATION
# ========================================
from pathlib import Path
import json

print('=' * 60)
print('EXPORTING PARQUET DATASETS WITH VALIDATION')
print('=' * 60)

manifest = {
    'export_tag': EXPORT_TAG,
    'export_root': str(export_root),
    'created_utc': datetime.utcnow().isoformat(),
    'datasets': {},
    'segmentation_support': {},
    'validation_results': []
}

for name, df in datasets.items():
    print(f'\n[{name}]')
    dataset_path = export_root / name
    export_df = df
    
    # Validate schema completeness
    validation_result, segment_info = validate_schema_completeness(df, name)
    manifest['validation_results'].append(validation_result)
    
    # Add month_period if time column exists
    time_col = TIME_COLUMN_MAP.get(name)
    if time_col and time_col in df.columns:
        export_df = add_month_period(export_df, time_col)
        print(f'  Added month_period from {time_col}')
    
    # Export the dataset
    export_dataset(export_df, dataset_path, overwrite=OVERWRITE)
    print(f'  ✓ Wrote parquet: {dataset_path}')
    
    # Get column list for manifest
    column_list = [field.name for field in export_df.schema.fields]
    
    # Print schema preview and segmentation info
    print(f'  ✓ Schema: {preview_schema(export_df)}')
    if segment_info['has_segmentation']:
        if segment_info['source_columns']:
            print(f'  ✓ Segment source columns: {", ".join(segment_info["source_columns"])}')
        if segment_info['dimension_columns']:
            print(f'  ✓ Segment dimension columns: {", ".join(segment_info["dimension_columns"])}')
    
    # Add to manifest with enhanced metadata
    manifest['datasets'][name] = {
        'source_table': TABLES[name],
        'parquet_path': str(dataset_path),
        'has_month_period': 'month_period' in export_df.columns,
        'schema_preview': preview_schema(export_df),
        'time_column': time_col,
        'column_count': len(column_list),
        'columns': column_list,
        'segmentation': segment_info
    }
    
    # Track segmentation support
    manifest['segmentation_support'][name] = segment_info['has_segmentation']
    
    # Print validation issues if any
    if validation_result['issues']:
        for issue in validation_result['issues']:
            print(f'  ⚠ {issue}')

# Write manifest
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('\n✓ Manifest written with enhanced metadata')
print(f'  {manifest_path}')
print('=' * 60)

EXPORTING PARQUET DATASETS WITH VALIDATION

[raw_sample]
  Added month_period from datestamp


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/raw_sample
  ✓ Schema: datestamp:timestamp, tensor_util:double, gpu_util:double, tensor_tflops:double, tensor_tflops_sm:double, chip_power:double, redfish_power:double, sm_active:double, dram_active:double, mem_copy_util:double, vram_usage:double, sm_clock:double, ... (30 total columns)
  ✓ Segment source columns: gen, region_rollup, customer_segment, product_segment, is_training
  ✓ Segment dimension columns: segment_dim, segment_value

[corr_summary]


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/corr_summary
  ✓ Schema: metric_x:string, metric_y:string, base_metric_x:string, base_metric_y:string, method:string, summary_percentile:string, summary_feature_group:string, correlation_value:double, obs_count:bigint, run_id:string, run_timestamp:timestamp, summary_time_col:string, ... (14 total columns)
  ✓ Segment dimension columns: segment_dim, segment_value

[corr_over_time]
  Added month_period from time_bucket


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/corr_over_time
  ✓ Schema: time_bucket:timestamp, metric_x:string, metric_y:string, base_metric_x:string, base_metric_y:string, correlation_value:double, method:string, summary_percentile:string, run_id:string, time_grain:string, segment_dim:string, segment_value:string, ... (13 total columns)
  ✓ Segment dimension columns: segment_dim, segment_value

[density_bins]


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/density_bins
  ✓ Schema: metric_x:string, metric_y:string, x_bin:int, y_bin:int, count:bigint, segment_dim:string, segment_value:string
  ✓ Segment dimension columns: segment_dim, segment_value

[density_over_time]
  Added month_period from time_bucket


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/density_over_time
  ✓ Schema: time_bucket:timestamp, metric_x:string, metric_y:string, x_bin:int, y_bin:int, count:bigint, segment_dim:string, segment_value:string, month_period:string
  ✓ Segment dimension columns: segment_dim, segment_value

[bin_metadata]


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/bin_metadata
  ✓ Schema: metric_name:string, bin_number:int, bin_min:double, bin_max:double, num_bins:int, percentile_low:double, percentile_high:double, run_id:string, segment_dim:string, segment_value:string
  ✓ Segment dimension columns: segment_dim, segment_value

[trend_summary]
  Added month_period from time_bucket


  ✓ Wrote parquet: /home/coreweave/dcgm_eda_parquet/20260625_164113/trend_summary
  ✓ Schema: time_bucket:timestamp, metric_name:string, base_metric:string, summary_feature_group:string, observation_count:bigint, mean_value:double, median_value:double, run_id:string, segment_dim:string, segment_value:string, month_period:string
  ✓ Segment dimension columns: segment_dim, segment_value

✓ Manifest written with enhanced metadata
  /home/coreweave/dcgm_eda_parquet/20260625_164113/manifest.json


In [18]:
# ========================================
# DOWNSTREAM READINESS SUMMARY
# ========================================
print('=' * 60)
print('DOWNSTREAM READINESS SUMMARY')
print('=' * 60)

print('\n📋 Export Contract Summary:')
print(f'  Export Root: {export_root}')
print(f'  Manifest: {manifest_path}')
print(f'  Compression: {COMPRESSION}')
print(f'  Export Tag: {EXPORT_TAG}')

print('\n📊 Dataset Export Status:')
for name, meta in manifest['datasets'].items():
    seg_marker = '✓' if meta['segmentation']['has_segmentation'] else '○'
    time_marker = '⏰' if meta['has_month_period'] else ' '
    print(f'  {seg_marker}{time_marker} {name}: {meta["column_count"]} columns')

print('\n🔍 What the Local Visualization Notebook Can Assume:')
print('  1. All datasets are exported as Parquet with snappy compression')
print('  2. manifest.json contains:')
print('     - Dataset paths and column lists')
print('     - Segmentation metadata (which columns are available)')
print('     - Time column mappings')
print('  3. Time-based datasets include month_period for monthly charts')
print('  4. Segmentation columns (when present):')
print('     - raw_sample: may contain gen, region_rollup, customer_segment, product_segment, is_training')
print('     - other datasets: may contain segment_dim and segment_value')

print('\n⚠️  Schema Changes to Handle Downstream:')
if any(not supported for supported in manifest['segmentation_support'].values()):
    print('  - Segmentation columns not yet present in all datasets')
    print('  - Local visualization should handle both segmented and non-segmented data')
else:
    print('  - All datasets have segmentation support')
    print('  - Local visualization can rely on segment columns')

print('\n✅ Next Steps:')
print('  1. Open dcgm_eda_local_visualizations_monthly_1.0.0.ipynb')
print(f'  2. Set PARQUET_ROOT = "{export_root}"')
print('  3. The notebook will auto-discover datasets via manifest.json')
print('  4. Segmented visualizations will be available for datasets with segment columns')

print('=' * 60)

DOWNSTREAM READINESS SUMMARY

📋 Export Contract Summary:
  Export Root: /home/coreweave/dcgm_eda_parquet/20260625_164113
  Manifest: /home/coreweave/dcgm_eda_parquet/20260625_164113/manifest.json
  Compression: snappy
  Export Tag: 20260625_164113

📊 Dataset Export Status:
  ✓⏰ raw_sample: 30 columns
  ✓  corr_summary: 14 columns
  ✓⏰ corr_over_time: 13 columns
  ✓  density_bins: 7 columns
  ✓⏰ density_over_time: 9 columns
  ✓  bin_metadata: 10 columns
  ✓⏰ trend_summary: 11 columns

🔍 What the Local Visualization Notebook Can Assume:
  1. All datasets are exported as Parquet with snappy compression
  2. manifest.json contains:
     - Dataset paths and column lists
     - Segmentation metadata (which columns are available)
     - Time column mappings
  3. Time-based datasets include month_period for monthly charts
  4. Segmentation columns (when present):
     - raw_sample: may contain gen, region_rollup, customer_segment, product_segment, is_training
     - other datasets: may contain

In [19]:
# ========================================
# FINAL CLEANUP
# ========================================

print('=' * 60)
print('FINAL CLEANUP')
print('=' * 60)

for df_name in ['df_raw_sample', 'df_summary_clean', '_df_binned', '_df_binned_time']:
    if df_name in globals():
        try:
            globals()[df_name].unpersist()
            print(f'✓ Unpersisted {df_name}')
        except Exception as e:
            print(f'Note: could not unpersist {df_name}: {e}')

try:
    spark.catalog.clearCache()
    print('✓ Cleared Spark cache')
except Exception as e:
    print(f'Note: could not clear Spark cache: {e}')

print('=' * 60)
print('✓ CLEANUP COMPLETE')
print('=' * 60)

FINAL CLEANUP
✓ Cleared Spark cache
✓ CLEANUP COMPLETE


In [20]:
# ========================================
# FINAL CLEANUP
# ========================================

print("="*60)
print("FINAL CLEANUP")
print("="*60)

for df_name in ["df_raw_sample", "df_summary_clean", "_df_binned", "_df_binned_time"]:
    if df_name in globals():
        try:
            globals()[df_name].unpersist()
            print(f"✓ Unpersisted {df_name}")
        except Exception as e:
            print(f"Note: could not unpersist {df_name}: {e}")

try:
    spark.catalog.clearCache()
    print("✓ Cleared Spark cache")
except Exception as e:
    print(f"Note: could not clear Spark cache: {e}")

print("="*60)
print("✓ CLEANUP COMPLETE")
print("="*60)


FINAL CLEANUP
✓ Cleared Spark cache
✓ CLEANUP COMPLETE
